# E12 Integración y verificación del pipeline local

Laboratorio de referencia para las sesiones del 30 y 31 de octubre de 2026. Integra recepción, validación, transformación, publicación candidata, consulta SQL, reejecución y auditoría sobre las tres muestras PQRS incluidas en el repositorio.

**Pregunta:** ¿cuántos reportes contiene cada corte por territorio del afectado?

**Unidad de observación:** una fila de reporte de los archivos seleccionados. Un reporte no equivale a una persona única ni permite estimar prevalencia.

**Productos:** detalle aceptado, registros rechazados con motivo, agregado por corte y territorio, dos registros operativos, comparación de reejecuciones y ficha de auditoría.

## 1. Entorno, rutas y configuración

El laboratorio usa Python, pandas, PyArrow y DuckDB. Detecta la raíz del repositorio para funcionar desde `soluciones/`, `kit/` o la raíz. Las salidas regenerables quedan en `kit/salidas/soluciones/E12`, carpeta excluida de Git.

In [1]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import hashlib
import json
import platform

import duckdb
import pandas as pd
import pyarrow
from IPython.display import display


def encontrar_repositorio(inicio=None):
    inicio = Path.cwd() if inicio is None else Path(inicio)
    for candidato in (inicio, *inicio.parents):
        if (candidato / 'kit' / 'pqrs_fuentes.json').is_file():
            return candidato
        anidado = candidato / 'big-data-postgraduate'
        if (anidado / 'kit' / 'pqrs_fuentes.json').is_file():
            return anidado
    raise FileNotFoundError('No se encontró la raíz de big-data-postgraduate')


REPOSITORIO = encontrar_repositorio()
DATOS = REPOSITORIO / 'kit' / 'data' / 'pqrs_muestras'
SALIDA = REPOSITORIO / 'kit' / 'salidas' / 'soluciones' / 'E12'
MANIFIESTO_RUTA = REPOSITORIO / 'kit' / 'pqrs_fuentes.json'
SALIDA.mkdir(parents=True, exist_ok=True)

CONFIGURACION = {
    'producto': 'conteo_reportes_por_corte_y_territorio_afectado',
    'version_reglas': 'e12-pqrs-v1',
    'periodos': ['2023_II', '2024_I', '2024_II'],
    'dimensiones_agregado': ['corte_archivo', 'territorio'],
    'medida': 'reportes',
    'zona_horaria': 'America/Bogota',
}

print('Repositorio:', REPOSITORIO)
print('Datos:', DATOS)
print('Salidas:', SALIDA)
print('Python:', platform.python_version())
print('pandas:', pd.__version__, '| pyarrow:', pyarrow.__version__, '| duckdb:', duckdb.__version__)

Repositorio: /Users/eshernan/Library/CloudStorage/OneDrive-Personal/Documentos/UPTC/Formats/big-data-postgraduate
Datos: /Users/eshernan/Library/CloudStorage/OneDrive-Personal/Documentos/UPTC/Formats/big-data-postgraduate/kit/data/pqrs_muestras
Salidas: /Users/eshernan/Library/CloudStorage/OneDrive-Personal/Documentos/UPTC/Formats/big-data-postgraduate/kit/salidas/soluciones/E12
Python: 3.12.14
pandas: 2.2.3 | pyarrow: 19.0.1 | duckdb: 1.4.1


## 2. Funciones de control y firma

Las funciones conservan el procedimiento visible: verifican archivos y columnas, calculan una firma estable del contenido y aplican reglas por fila. Una fila rechazada se cuenta una sola vez, aunque registre varios motivos.

In [2]:
def sha256_archivo(ruta, bloque=1024 * 1024):
    firma = hashlib.sha256()
    with Path(ruta).open('rb') as archivo:
        for fragmento in iter(lambda: archivo.read(bloque), b''):
            firma.update(fragmento)
    return firma.hexdigest()


def firma_dataframe(datos, columnas, orden):
    normalizado = datos.loc[:, columnas].copy()
    normalizado = normalizado.sort_values(orden, kind='stable').reset_index(drop=True)
    normalizado = normalizado.astype('string').fillna('<NA>')
    texto = normalizado.to_csv(index=False, lineterminator='\n')
    return hashlib.sha256(texto.encode('utf-8')).hexdigest()


def validar_contrato(datos, columnas_esperadas, nombre):
    actuales = datos.columns.tolist()
    if actuales != columnas_esperadas:
        faltantes = sorted(set(columnas_esperadas) - set(actuales))
        extras = sorted(set(actuales) - set(columnas_esperadas))
        raise ValueError(
            f'Contrato incumplido en {nombre}; faltantes={faltantes}; extras={extras}; '
            'también debe conservarse el orden de columnas'
        )


def agregar_motivo(serie, mascara, motivo):
    previo = serie.loc[mascara]
    serie.loc[mascara] = previo.where(previo.eq(''), previo + '|') + motivo


def validar_y_transformar(datos):
    trabajo = datos.copy()
    objectid = trabajo['OBJECTID'].astype('string').fillna('').str.strip()
    periodo_num = pd.to_numeric(trabajo['periodo'].astype('string').str.strip(), errors='coerce')
    mes_num = pd.to_numeric(trabajo['mes'].astype('string').str.strip(), errors='coerce')
    territorio_raw = trabajo['afec_cod_mpio'].astype('string').fillna('').str.strip()

    motivos = pd.Series('', index=trabajo.index, dtype='string')
    agregar_motivo(motivos, objectid.eq(''), 'OBJECTID_VACIO')
    agregar_motivo(motivos, periodo_num.isna() | periodo_num.mod(1).ne(0), 'PERIODO_INVALIDO')
    agregar_motivo(motivos, mes_num.isna() | mes_num.mod(1).ne(0) | ~mes_num.between(1, 12), 'MES_INVALIDO')
    territorio_invalido = territorio_raw.ne('') & ~territorio_raw.str.fullmatch(r'[0-9]{1,5}', na=False)
    agregar_motivo(motivos, territorio_invalido, 'TERRITORIO_INVALIDO')

    rechazado = motivos.ne('')
    rechazados = trabajo.loc[rechazado, [
        'corte_archivo', 'archivo_fuente', 'fila_fuente', 'OBJECTID',
        'periodo', 'mes', 'afec_cod_mpio'
    ]].copy()
    rechazados['motivos_rechazo'] = motivos.loc[rechazado].to_numpy()

    aceptados = trabajo.loc[~rechazado].copy()
    aceptados['periodo_fuente'] = periodo_num.loc[~rechazado].astype('Int64').to_numpy()
    aceptados['mes_num'] = mes_num.loc[~rechazado].astype('Int64').to_numpy()
    territorio_aceptado = territorio_raw.loc[~rechazado]
    aceptados['territorio'] = territorio_aceptado.where(
        territorio_aceptado.ne(''), 'SIN_INFORMAR'
    ).str.zfill(5).to_numpy()
    aceptados.loc[aceptados['territorio'].eq('SIN_INFORMAR'), 'territorio'] = 'SIN_INFORMAR'
    aceptados['clave_reporte'] = (
        aceptados['corte_archivo'].astype(str) + '|' + aceptados['OBJECTID'].astype(str)
    )

    columnas_detalle = [
        'corte_archivo', 'archivo_fuente', 'fila_fuente', 'OBJECTID', 'clave_reporte',
        'periodo_fuente', 'mes_num', 'territorio', 'pqr_canal', 'ent_nombre'
    ]
    detalle = aceptados[columnas_detalle].copy()
    agregado = (
        detalle.groupby(['corte_archivo', 'territorio'], dropna=False)
        .size().rename('reportes').reset_index()
        .sort_values(['corte_archivo', 'territorio']).reset_index(drop=True)
    )
    return detalle, rechazados, agregado

## 3. Recepción y verificación de las entradas

Cada archivo se lee con el separador declarado. Antes de integrarlo se verifican tamaño lógico, columnas y SHA-256 de la muestra. Los originales permanecen sin modificaciones.

In [3]:
with MANIFIESTO_RUTA.open(encoding='utf-8') as archivo:
    manifiesto = {item['periodo']: item for item in json.load(archivo)}

entradas = []
controles_entrada = []
for corte in CONFIGURACION['periodos']:
    referencia = manifiesto[corte]
    ruta = DATOS / f'Muestra_1000_{corte}.csv'
    huella = sha256_archivo(ruta)
    assert huella == referencia['sha256_muestra'], f'Huella inesperada: {ruta.name}'

    datos = pd.read_csv(
        ruta, sep=referencia['separador'], encoding=referencia['codificacion'],
        dtype=str, keep_default_na=False
    )
    validar_contrato(datos, referencia['columnas'], ruta.name)
    assert datos.shape == (referencia['filas_muestra'], referencia['cantidad_columnas'])

    datos = datos.assign(
        corte_archivo=corte,
        archivo_fuente=ruta.name,
        fila_fuente=range(2, len(datos) + 2),
        sha256_fuente=huella,
    )
    entradas.append(datos)
    controles_entrada.append({
        'corte': corte, 'archivo': ruta.name, 'filas': len(datos),
        'columnas': referencia['cantidad_columnas'], 'sha256': huella
    })

df_entrada = pd.concat(entradas, ignore_index=True)
assert len(df_entrada) == sum(item['filas'] for item in controles_entrada) == 3000
assert df_entrada[['corte_archivo', 'OBJECTID']].duplicated().sum() == 0

display(pd.DataFrame(controles_entrada))
print('Filas recibidas:', len(df_entrada))

,corte,archivo,filas,columnas,sha256
0,2023_II,Muestra_1000_2023_II.csv,1000,38,5546bb62ca598654a821c147d1efdc92f165324c13f904...
1,2024_I,Muestra_1000_2024_I.csv,1000,38,f06abcc52537eff223894edbeb83aedd9719d45b7d0887...
2,2024_II,Muestra_1000_2024_II.csv,1000,38,38102044d2c66b86b1e081867bbc854c4ea220524b51a0...


Filas recibidas: 3000


## 4. Caso didáctico resuelto

Antes de ejecutar las muestras reales se comprueban las reglas con seis filas. El resultado esperado es `6 = 5 + 1`, cuatro grupos publicados y un rechazo por período inválido. Esta prueba detecta errores que un total aislado ocultaría.

In [4]:
caso_seis_filas = pd.DataFrame({
    'corte_archivo': ['2024_I'] * 4 + ['2024_II'] * 2,
    'archivo_fuente': ['caso_didactico.csv'] * 6,
    'fila_fuente': range(2, 8),
    'OBJECTID': ['1', '2', '3', '4', '5', '6'],
    'periodo': ['2024', '2024', '2024', '2024', '2024', 'invalido'],
    'mes': ['1', '1', '1', '1', '7', '7'],
    'afec_cod_mpio': ['11001', '11001', '15001', '', '11001', '15001'],
    'pqr_canal': ['Web'] * 6,
    'ent_nombre': ['Entidad didáctica'] * 6,
})

detalle_caso, rechazados_caso, agregado_caso = validar_y_transformar(caso_seis_filas)
assert len(caso_seis_filas) == len(detalle_caso) + len(rechazados_caso) == 6
assert (len(detalle_caso), len(rechazados_caso), len(agregado_caso)) == (5, 1, 4)
assert rechazados_caso.iloc[0]['motivos_rechazo'] == 'PERIODO_INVALIDO'
assert agregado_caso['reportes'].sum() == 5

display(agregado_caso)
display(rechazados_caso[['fila_fuente', 'periodo', 'motivos_rechazo']])
print('Prueba superada: 6 leídas = 5 aceptadas + 1 rechazada; 4 grupos publicados')

,corte_archivo,territorio,reportes
0,2024_I,11001,2
1,2024_I,15001,1
2,2024_I,SIN_INFORMAR,1
3,2024_II,11001,1


,fila_fuente,periodo,motivos_rechazo
5,7,invalido,PERIODO_INVALIDO


Prueba superada: 6 leídas = 5 aceptadas + 1 rechazada; 4 grupos publicados


## 5. Pipeline integrado

Cada ejecución crea una carpeta candidata independiente. Después de escribir, relee los tres Parquet y usa DuckDB para reconstruir el agregado desde el detalle. Solo asigna estado `VALIDA` cuando concilian las filas, cada grupo y el total.

In [5]:
COLUMNAS_DETALLE = [
    'corte_archivo', 'archivo_fuente', 'fila_fuente', 'OBJECTID', 'clave_reporte',
    'periodo_fuente', 'mes_num', 'territorio', 'pqr_canal', 'ent_nombre'
]
COLUMNAS_RECHAZADOS = [
    'corte_archivo', 'archivo_fuente', 'fila_fuente', 'OBJECTID',
    'periodo', 'mes', 'afec_cod_mpio', 'motivos_rechazo'
]
COLUMNAS_AGREGADO = ['corte_archivo', 'territorio', 'reportes']


def ejecutar_pipeline(identificador, datos_entrada, controles_fuente):
    carpeta = SALIDA / 'candidatos' / identificador
    carpeta.mkdir(parents=True, exist_ok=True)
    rutas = {
        'detalle': carpeta / 'detalle.parquet',
        'rechazados': carpeta / 'rechazados.parquet',
        'agregado': carpeta / 'agregado.parquet',
        'consulta': carpeta / 'consulta_duckdb.csv',
        'registro': carpeta / 'registro.json',
    }

    inicio = datetime.now(ZoneInfo(CONFIGURACION['zona_horaria']))
    detalle, rechazados, agregado = validar_y_transformar(datos_entrada)
    assert len(datos_entrada) == len(detalle) + len(rechazados)
    assert agregado['reportes'].sum() == len(detalle)

    detalle.to_parquet(rutas['detalle'], index=False, compression='zstd')
    rechazados.to_parquet(rutas['rechazados'], index=False, compression='zstd')
    agregado.to_parquet(rutas['agregado'], index=False, compression='zstd')

    detalle_leido = pd.read_parquet(rutas['detalle'])
    rechazados_leidos = pd.read_parquet(rutas['rechazados'])
    agregado_leido = pd.read_parquet(rutas['agregado'])

    with duckdb.connect() as conexion:
        consulta = conexion.execute(
            '''
            SELECT corte_archivo, territorio, COUNT(*) AS reportes
            FROM read_parquet(?)
            GROUP BY corte_archivo, territorio
            ORDER BY corte_archivo, territorio
            ''',
            [str(rutas['detalle'])],
        ).df()
    consulta.to_csv(rutas['consulta'], index=False)

    pd.testing.assert_frame_equal(
        agregado_leido[COLUMNAS_AGREGADO].reset_index(drop=True),
        consulta[COLUMNAS_AGREGADO].reset_index(drop=True),
        check_dtype=False,
    )
    assert len(datos_entrada) == len(detalle_leido) + len(rechazados_leidos)
    assert consulta['reportes'].sum() == len(detalle_leido)

    firmas = {
        'detalle': firma_dataframe(
            detalle_leido, COLUMNAS_DETALLE, ['corte_archivo', 'fila_fuente']
        ),
        'rechazados': firma_dataframe(
            rechazados_leidos, COLUMNAS_RECHAZADOS, ['corte_archivo', 'fila_fuente']
        ),
        'agregado': firma_dataframe(
            agregado_leido, COLUMNAS_AGREGADO, ['corte_archivo', 'territorio']
        ),
    }
    fin = datetime.now(ZoneInfo(CONFIGURACION['zona_horaria']))
    registro = {
        'identificador': identificador,
        'estado': 'VALIDA',
        'producto': CONFIGURACION['producto'],
        'version_reglas': CONFIGURACION['version_reglas'],
        'inicio': inicio.isoformat(),
        'fin': fin.isoformat(),
        'entradas': controles_fuente,
        'conteos': {
            'leidas': len(datos_entrada),
            'aceptadas': len(detalle_leido),
            'rechazadas': len(rechazados_leidos),
            'grupos': len(agregado_leido),
            'total_agregado': int(agregado_leido['reportes'].sum()),
        },
        'controles': {
            'leidas_igual_aceptadas_mas_rechazadas': True,
            'agregado_igual_detalle_por_grupo': True,
            'agregado_igual_detalle_total': True,
            'originales_modificados': False,
        },
        'firmas_contenido': firmas,
        'rutas': {clave: str(valor.relative_to(REPOSITORIO)) for clave, valor in rutas.items()},
        'versiones': {
            'python': platform.python_version(), 'pandas': pd.__version__,
            'pyarrow': pyarrow.__version__, 'duckdb': duckdb.__version__,
        },
    }
    with rutas['registro'].open('w', encoding='utf-8') as archivo:
        json.dump(registro, archivo, ensure_ascii=False, indent=2)
    return registro, detalle_leido, rechazados_leidos, agregado_leido

## 6. Primera ejecución y publicación válida inicial

La ejecución A produce el primer candidato. El archivo `publicacion_valida.json` se actualiza únicamente después de comprobar su estado.

In [6]:
registro_a, detalle_a, rechazados_a, agregado_a = ejecutar_pipeline(
    'ejecucion_a', df_entrada, controles_entrada
)
assert registro_a['estado'] == 'VALIDA'

publicacion_ruta = SALIDA / 'publicacion_valida.json'
publicacion = {
    'ejecucion_valida': 'ejecucion_a',
    'estado': 'VALIDA',
    'actualizada': datetime.now(ZoneInfo(CONFIGURACION['zona_horaria'])).isoformat(),
    'motivo': 'Primera ejecución conciliada por grupo y total',
}
with publicacion_ruta.open('w', encoding='utf-8') as archivo:
    json.dump(publicacion, archivo, ensure_ascii=False, indent=2)

display(pd.DataFrame([registro_a['conteos']]))
display(agregado_a.head(10))
print('Publicación válida:', publicacion['ejecucion_valida'])

,leidas,aceptadas,rechazadas,grupos,total_agregado
0,3000,3000,0,687,3000


,corte_archivo,territorio,reportes
0,2023_II,05001,85
1,2023_II,05034,1
2,2023_II,05040,1
3,2023_II,05045,2
4,2023_II,05051,1
5,2023_II,05079,1
6,2023_II,05088,18
7,2023_II,05129,4
8,2023_II,05142,1
9,2023_II,05154,2


Publicación válida: ejecucion_a


## 7. Segunda ejecución y comparación del contenido

La ejecución B usa exactamente las mismas entradas y reglas, escribe en otra carpeta y no anexa resultados anteriores. Los identificadores y tiempos pueden cambiar; las firmas del detalle, rechazados y agregado deben permanecer iguales.

In [7]:
registro_b, detalle_b, rechazados_b, agregado_b = ejecutar_pipeline(
    'ejecucion_b', df_entrada, controles_entrada
)

comparacion = {
    producto: registro_a['firmas_contenido'][producto] == registro_b['firmas_contenido'][producto]
    for producto in ['detalle', 'rechazados', 'agregado']
}
comparacion['conteos_iguales'] = registro_a['conteos'] == registro_b['conteos']
comparacion['estable'] = all(comparacion.values())
assert comparacion['estable'], comparacion

pd.testing.assert_frame_equal(detalle_a, detalle_b)
pd.testing.assert_frame_equal(rechazados_a, rechazados_b)
pd.testing.assert_frame_equal(agregado_a, agregado_b)

comparacion_documento = {
    'ejecucion_a': registro_a['identificador'],
    'ejecucion_b': registro_b['identificador'],
    'comparaciones': comparacion,
    'firmas_a': registro_a['firmas_contenido'],
    'firmas_b': registro_b['firmas_contenido'],
}
with (SALIDA / 'comparacion_reejecuciones.json').open('w', encoding='utf-8') as archivo:
    json.dump(comparacion_documento, archivo, ensure_ascii=False, indent=2)

# Solo ahora la ejecución B reemplaza a A como publicación identificada.
publicacion = {
    'ejecucion_valida': 'ejecucion_b',
    'ejecucion_anterior': 'ejecucion_a',
    'estado': 'VALIDA',
    'actualizada': datetime.now(ZoneInfo(CONFIGURACION['zona_horaria'])).isoformat(),
    'motivo': 'Segunda ejecución estable y conciliada; se conserva la ejecución anterior',
}
with publicacion_ruta.open('w', encoding='utf-8') as archivo:
    json.dump(publicacion, archivo, ensure_ascii=False, indent=2)

display(pd.DataFrame([comparacion]))
print('Publicación válida:', publicacion['ejecucion_valida'])

,detalle,rechazados,agregado,conteos_iguales,estable
0,True,True,True,True,True


Publicación válida: ejecucion_b


## 8. Fallo controlado de contrato

Retirar una columna obligatoria debe detener el proceso antes de publicar. La prueba ocurre solo en memoria; los archivos originales y la publicación válida permanecen intactos.

In [8]:
entrada_defectuosa = entradas[0].drop(columns=['mes'])
try:
    validar_contrato(
        entrada_defectuosa.drop(columns=['corte_archivo', 'archivo_fuente', 'fila_fuente', 'sha256_fuente']),
        manifiesto['2023_II']['columnas'],
        'entrada_defectuosa_en_memoria',
    )
except ValueError as error:
    incidencia_controlada = {
        'estado': 'DETENIDA',
        'causa_observada': str(error),
        'accion': 'No publicar; restaurar desde el original verificado y volver a ejecutar',
        'publicacion_conservada': publicacion['ejecucion_valida'],
    }
else:
    raise AssertionError('El contrato debía detener la entrada sin la columna mes')

with (SALIDA / 'incidencia_contrato_controlada.json').open('w', encoding='utf-8') as archivo:
    json.dump(incidencia_controlada, archivo, ensure_ascii=False, indent=2)

display(incidencia_controlada)

{'estado': 'DETENIDA',
 'causa_observada': "Contrato incumplido en entrada_defectuosa_en_memoria; faltantes=['mes']; extras=[]; también debe conservarse el orden de columnas",
 'accion': 'No publicar; restaurar desde el original verificado y volver a ejecutar',
 'publicacion_conservada': 'ejecucion_b'}

## 9. Auditoría de referencia

Esta ficha demuestra los controles del laboratorio. En la actividad estudiantil, otro equipo debe repetir una consulta y registrar su nombre, evidencia y hallazgo sin inventar defectos.

In [9]:
auditoria = pd.DataFrame([
    {
        'control': 'Entradas identificadas', 'estado': 'CUMPLE',
        'evidencia': 'Tres archivos, cortes y SHA-256 en cada registro',
        'accion': 'Ninguna',
    },
    {
        'control': 'Conteos conciliados', 'estado': 'CUMPLE',
        'evidencia': f"{registro_b['conteos']['leidas']} = {registro_b['conteos']['aceptadas']} + {registro_b['conteos']['rechazadas']}",
        'accion': 'Ninguna',
    },
    {
        'control': 'Agregado contra detalle', 'estado': 'CUMPLE',
        'evidencia': 'DuckDB reprodujo cada grupo y el total del agregado',
        'accion': 'Ninguna',
    },
    {
        'control': 'Reejecución estable', 'estado': 'CUMPLE',
        'evidencia': 'Firmas de detalle, rechazados y agregado iguales entre A y B',
        'accion': 'Ninguna',
    },
    {
        'control': 'Límite de interpretación', 'estado': 'CUMPLE',
        'evidencia': 'Los conteos representan reportes de las muestras, no personas ni prevalencia',
        'accion': 'Mantener el alcance junto al producto',
    },
])
auditoria.to_csv(SALIDA / 'ficha_auditoria_referencia.csv', index=False)
display(auditoria)
assert auditoria['estado'].eq('CUMPLE').all()

,control,estado,evidencia,accion
0,Entradas identificadas,CUMPLE,"Tres archivos, cortes y SHA-256 en cada registro",Ninguna
1,Conteos conciliados,CUMPLE,3000 = 3000 + 0,Ninguna
2,Agregado contra detalle,CUMPLE,DuckDB reprodujo cada grupo y el total del agr...,Ninguna
3,Reejecución estable,CUMPLE,"Firmas de detalle, rechazados y agregado igual...",Ninguna
4,Límite de interpretación,CUMPLE,Los conteos representan reportes de las muestr...,Mantener el alcance junto al producto


## 10. Resultado y criterios de aceptación

El laboratorio queda aceptado cuando se cumplen simultáneamente: ejecución completa, conteos explicables, agregado conciliado con el detalle y reejecución estable. El registro identifica la publicación válida y conserva la ejecución anterior.

Las muestras contienen las primeras 1.000 filas de cada corte y no son aleatorias. Los resultados no deben extrapolarse al universo de PQRS ni interpretarse como personas únicas.

In [10]:
criterios_finales = {
    'ejecucion_completa': registro_b['estado'] == 'VALIDA',
    'conteos_explicables': (
        registro_b['conteos']['leidas']
        == registro_b['conteos']['aceptadas'] + registro_b['conteos']['rechazadas']
    ),
    'agregado_consistente': registro_b['controles']['agregado_igual_detalle_por_grupo'],
    'reejecucion_estable': comparacion['estable'],
}
assert all(criterios_finales.values())

resumen_final = {
    'criterios': criterios_finales,
    'publicacion_valida': publicacion['ejecucion_valida'],
    'conteos': registro_b['conteos'],
    'salida_relativa': str(SALIDA.relative_to(REPOSITORIO)),
}
with (SALIDA / 'resumen_final.json').open('w', encoding='utf-8') as archivo:
    json.dump(resumen_final, archivo, ensure_ascii=False, indent=2)

display(pd.DataFrame([criterios_finales]))
display(pd.DataFrame([registro_b['conteos']]))
print('E12 completado y verificado. Salidas:', resumen_final['salida_relativa'])

,ejecucion_completa,conteos_explicables,agregado_consistente,reejecucion_estable
0,True,True,True,True


,leidas,aceptadas,rechazadas,grupos,total_agregado
0,3000,3000,0,687,3000


E12 completado y verificado. Salidas: kit/salidas/soluciones/E12
